<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/demos/Week03_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 3** - Data Visualization

## Loading the data

In [ ]:
import pandas as pd

# below is the link to the GitHub data
# Description of the data is here https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00597/garments_worker_productivity.csv'
df = pd.read_csv(url)

df.head()

In [ ]:
# let's first check what are the columns and their types
df.info()

## Pre-processing the data

In [ ]:
# do a preliminary check of the unique values
quarter_list = df['quarter'].unique()
department_list = df['department'].unique()
day_list = df['day'].unique()
team_list = df['team'].unique()

print(quarter_list)
print(department_list)
print(day_list)
print(team_list)

In [ ]:
# there's spelling errors in the department data! Let's correct that
df.loc[df['department']=='finishing ', 'department'] = 'finishing'
df.loc[df['department']=='sweing', 'department'] = 'sewing'

## Converting date variables

In [ ]:
# Pre-processing the data

# Let's convert the date into a Python date format
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')

### Fun with DateTime objects

In [ ]:
# Using pandas.to_datetime
# This already exists in your notebook from a previous attempt, showing how it works.
date_a = pd.to_datetime('2/3/2026', format='%m/%d/%Y')
print(f"Using pd.to_datetime with format: {date_a}")

# Without specifying a format, pandas often infers it
date_b = pd.to_datetime('2023-10-26', format='%Y-%m-%d')
print(f"Using pd.to_datetime with inference: {date_b}")

date_c = pd.to_datetime('2023', format='%Y')
print(f"Using pd.to_datetime with dayfirst: {date_c}")

print("How many days between Date A and Date B?")
num_days = (date_a - date_b).days
print(num_days)


## Converting categorical variables

In [ ]:
# convert the categorical values: quarter, department, day, team
df['quarter'] = df['quarter'].astype('category')
df['department'] = df['department'].astype('category')
df['day'] = df['day'].astype('category')
df['team'] = df['team'].astype('category')

df.info()

## Histograms

In [ ]:
import seaborn as sns

sns.histplot(data=df, x='actual_productivity', kde=True)

In [ ]:
sns.histplot(data=df, x='actual_productivity', hue='department', palette="colorblind")


## Scatter plot

In [ ]:
sns.relplot(data=df, x="actual_productivity", y="targeted_productivity",
            kind="scatter")

In [ ]:
sns.relplot(data=df, x="actual_productivity", y="targeted_productivity",
            kind="scatter", hue="department")

In [ ]:
sns.jointplot(data=df, x="actual_productivity",
              y="targeted_productivity", hue="department",
              kind="kde")

In [ ]:
desired_vars = ['actual_productivity', 'targeted_productivity',
                'over_time', 'smv', 'incentive',
                'idle_men', 'idle_time', 'wip',
                'no_of_style_change', 'no_of_workers']

# define the pair grid
g = sns.PairGrid(df[desired_vars])
# map on the pair grid a scatter plot
g.map(sns.scatterplot)

In [ ]:
corr_matrix = df[desired_vars].corr()

sns.heatmap(corr_matrix, annot=False,
            cmap="vlag",
            vmin=-1,
            vmax=1)

## Line plot

In [ ]:
sns.relplot(df, x="date", y="actual_productivity", kind="line")

In [ ]:
g = sns.relplot(df, x="date", y="actual_productivity",
            kind="line", hue="department")
# Rotate x-axis labels for better readability
g.set_xticklabels(rotation=45)

In [ ]:
sns.relplot(df, x="date", y="actual_productivity",
                kind="line", hue="department",
                col="team",col_wrap=4
            )


In [ ]:
# What if I want to plot the total over_time by month, to see how it changes over time?

# first create a month column
df['month'] = df['date'].dt.month

# then, do a groupby month
df_to_plot = df.groupby(['month','department'])['over_time'].sum()

# this returns a pandas Series
# we need to convert this to a DataFrame
df_to_plot = df_to_plot.to_frame(name="total_overtime")

sns.relplot(df_to_plot, x='month', y='total_overtime', hue='department', kind='line')

## Categorical scatterplot

In [ ]:
sns.catplot(df, x="day", y="actual_productivity")

In [ ]:
sns.stripplot(data=df, x="day", y="actual_productivity", jitter=0.2, s=3)

## Box plot

In [ ]:
# display the "outliers"
Q3 = df['idle_time'].quantile(0.75)
Q2 = df['idle_time'].quantile(0.5)
Q1 = df['idle_time'].quantile(0.25)
IQR = Q3 - Q1
upper_whisker = Q3 + 1.5*IQR
lower_whisker = Q1 - 1.5*IQR

outliers = df[(df['idle_time'] > upper_whisker) | (df['idle_time'] < lower_whisker)]
display(outliers)

In [ ]:
sns.catplot(df, x="team", y="actual_productivity", kind="box")

In [ ]:
sns.catplot(df, x="team", y="actual_productivity", kind="box", hue="department")

In [ ]:
sns.catplot(df, x="day", y="actual_productivity",
            hue="quarter", kind="box")

In [ ]:
# zero-inflated feature
sns.catplot(df, y="idle_time", kind="box")

## Bar chart

In [ ]:
sns.catplot(df, x='team', y='actual_productivity', kind="bar")

In [ ]:
sns.catplot(df, x='team', y='actual_productivity', kind="bar", hue="department")

## Count plot

In [ ]:
sns.catplot(df, x="day", kind="count")

In [ ]:
sns.catplot(df, x="day", kind="count", hue="day", palette="colorblind")

## Customizing using matplotlib

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# g is a FacetGrid that has Matplotlib fig and ax as attributes
g = sns.catplot(data=df, x="team", y="actual_productivity",
            kind="box", hue="team", legend=False)

# Access the underlying Matplotlib Figure object and set its size
g.fig.set_size_inches(8, 5)

# matplotlib customizes it
g.ax.set_title('Productivity by team',
               fontsize=16, fontweight='bold')
g.ax.set_xlabel('Team', fontsize=13)
g.ax.set_ylabel('Actual Productivity', fontsize=13)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Plot 1: Histogram of actual_productivity
sns.histplot(data=df, x='actual_productivity', kde=True, ax=axes[0])
axes[0].set_title('Actual Productivity Distribution')

# Plot 2: Boxplot of actual_productivity by department
sns.boxplot(data=df, x='department', y='actual_productivity', ax=axes[1])
axes[1].set_title('Productivity by Department')

# Plot 3: Scatter plot of over_time vs actual_productivity
sns.scatterplot(data=df, x='over_time', y='actual_productivity', hue='department', ax=axes[2])
axes[2].set_title('Overtime vs Actual Productivity')

# Plot 4: Line plot of actual_productivity over time (average per day)
# First, calculate daily average for a cleaner line plot
daily_avg_prod = df.groupby('date')['actual_productivity'].mean().reset_index()
sns.lineplot(data=daily_avg_prod, x='date', y='actual_productivity', ax=axes[3])
axes[3].set_title('Average Daily Actual Productivity')
axes[3].tick_params(axis='x', rotation=45)
